# Embedding model test on your own data
Compares **Qwen3-Embedding-0.6B**, **harrier-oss-v1-0.6b** and **BGE-M3** on your documents and questions.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

**You need two CSV files:**
- `chunks.csv` with columns `chunk_id`, `text` (your document chunks)
- `eval.csv` with columns `query`, `relevant_chunk_id` (the chunk that answers the question; separate several ids with `;`)

No files yet? Skip the upload cell and run the demo-data cell instead.

In [ ]:
import torch
torch.is_cuda

## 1. Upload your CSVs

In [1]:
from google.colab import files
uploaded = files.upload()   # select chunks.csv and eval.csv

ModuleNotFoundError: No module named 'google.colab'

## 1b. (Optional) Demo data, only if you have no files yet

In [2]:
import pandas as pd
pd.DataFrame({
    "chunk_id": ["c1", "c2", "c3", "c4", "c5"],
    "text": [
        "To reset your password, open Settings, go to Security and click 'Reset password'.",
        "Refunds are available within 30 days of purchase if the product is unused.",
        "Our card works in over 150 countries with no foreign transaction fees.",
        "Creators are paid within 14 days after the sponsored video goes live.",
        "Para cancelar sua assinatura, acesse Conta > Plano e clique em Cancelar.",
    ],
}).to_csv("chunks.csv", index=False)
pd.DataFrame({
    "query": [
        "I forgot my login, how do I change it?",
        "can I get my money back",
        "does the card charge fees abroad",
        "when do influencers get their payment",
        "how do I cancel my subscription",
    ],
    "relevant_chunk_id": ["c1", "c2", "c3", "c4", "c5"],
}).to_csv("eval.csv", index=False)
print("Demo files created")

Demo files created


## 2. Load data

In [3]:
import pandas as pd
chunks = pd.read_csv("chunks.csv")
evalset = pd.read_csv("eval.csv")

chunks["chunk_id"] = chunks["chunk_id"].astype(str).str.strip()
evalset["relevant"] = evalset["relevant_chunk_id"].astype(str).apply(
    lambda s: {x.strip() for x in s.split(";") if x.strip()})

missing = set().union(*evalset["relevant"]) - set(chunks["chunk_id"])
print(f"{len(chunks)} chunks, {len(evalset)} queries")
if missing:
    print("WARNING: these relevant ids are not in chunks.csv:", missing)

5 chunks, 5 queries


## 3. Models to compare
Each model has its own query instruction (`prompt_name`). Documents are encoded without one.

In [4]:
MODELS = {
    "Qwen3-Embedding-0.6B": {"id": "Qwen/Qwen3-Embedding-0.6B",     "query_prompt": "query"},
    "harrier-oss-v1-0.6b":  {"id": "microsoft/harrier-oss-v1-0.6b", "query_prompt": "web_search_query"},
    "BGE-M3 (baseline)":    {"id": "BAAI/bge-m3",                   "query_prompt": None},
}
MAX_SEQ_LEN = 512   # raise if your chunks are longer than ~400 words

## 4. Evaluation function
- **Hit@k**: share of questions where the right chunk is in the top k results
- **MRR@10**: rewards putting the right chunk near the top (1.0 = always rank 1)

In [5]:
import torch, time, gc
from sentence_transformers import SentenceTransformer

KS = (1, 3, 5, 10)
ids = chunks["chunk_id"].values

def evaluate(name, cfg):
    model = SentenceTransformer(cfg["id"], device="cuda",model_kwargs={"torch_dtype": torch.float16})
    model.max_seq_length = MAX_SEQ_LEN

    t = time.time()
    doc_emb = model.encode(chunks["text"].astype(str).tolist(), batch_size=32,normalize_embeddings=True, convert_to_tensor=True,show_progress_bar=True)
    q_kwargs = {"prompt_name": cfg["query_prompt"]} if cfg["query_prompt"] else {}
    q_emb = model.encode(evalset["query"].astype(str).tolist(), batch_size=32,normalize_embeddings=True, convert_to_tensor=True, **q_kwargs)
    secs = time.time() - t

    k_max = min(max(KS), len(chunks))
    top = torch.topk(q_emb @ doc_emb.T, k=k_max, dim=1).indices.cpu().numpy()

    hits = {k: 0 for k in KS}
    mrr = 0.0
    misses = []
    for i, rel in enumerate(evalset["relevant"]):
        ranked = ids[top[i]]
        for k in KS:
            hits[k] += any(r in rel for r in ranked[:k])
        for rank, r in enumerate(ranked, 1):
            if r in rel:
                mrr += 1 / rank
                break
        if not any(r in rel for r in ranked[:5]):
            misses.append((evalset["query"].iloc[i], list(rel), list(ranked[:3])))

    n = len(evalset)
    result = {"model": name,
              **{f"Hit@{k}": round(hits[k] / n, 3) for k in KS},
              "MRR@10": round(mrr / n, 3),
              "embed_seconds": round(secs, 1)}

    del model, doc_emb, q_emb
    gc.collect(); torch.cuda.empty_cache()
    return result, misses

d:\AIMLPROJECT\Rag\basic_rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Run all models

In [6]:
results, all_misses = [], {}
for name, cfg in MODELS.items():
    print(f"\n=== {name} ===")
    res, misses = evaluate(name, cfg)
    results.append(res)
    all_misses[name] = misses

pd.DataFrame(results).sort_values("MRR@10", ascending=False).reset_index(drop=True)


=== Qwen3-Embedding-0.6B ===


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]



=== harrier-oss-v1-0.6b ===


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]



=== BGE-M3 (baseline) ===


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.64it/s]


,model,Hit@1,Hit@3,Hit@5,Hit@10,MRR@10,embed_seconds
0,Qwen3-Embedding-0.6B,1.0,1.0,1.0,1.0,1.0,1.7
1,harrier-oss-v1-0.6b,1.0,1.0,1.0,1.0,1.0,0.9
2,BGE-M3 (baseline),1.0,1.0,1.0,1.0,1.0,0.5


## 6. Where each model failed (right chunk not in top 5)
Read these: they show *why* a model loses, e.g. exact names, numbers, or language.

In [7]:
for name, misses in all_misses.items():
    print(f"\n=== {name}: {len(misses)} misses ===")
    for q, rel, got in misses[:10]:
        print(f"- {q}\n    expected {rel}, got {got}")


=== Qwen3-Embedding-0.6B: 0 misses ===

=== harrier-oss-v1-0.6b: 0 misses ===

=== BGE-M3 (baseline): 0 misses ===
